In [76]:
import numpy as np
import pandas as pd
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt
import hcp_utils as hcp
from matplotlib.colors import ListedColormap
import seaborn as sns
import os.path as op

bids_folder ='/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'
net_folder = op.join(bids_folder, 'derivatives', 'networks_infomap_full_01')
plots_folder = op.join(bids_folder, 'plots_and_ims', 'paper_01')

from numrisk.fmri_analysis.gradients.utils import get_basic_mask
mask, labeling_noParcel = get_basic_mask()

subList = [f'{sub:02d}' for sub in range(1, 67)]
group_assignment = pd.read_csv(op.join(bids_folder, 'group_assignment.csv')).set_index('subject')

In [21]:
from numrisk.fmri_analysis.gradients.utils import get_glasser_CAatlas_mapping

_, CA_names = get_glasser_CAatlas_mapping()

In [73]:
net_folder = op.join(bids_folder, 'derivatives', 'networks_infomap_full_01')

from numrisk.fmri_analysis.gradients.utils import get_glasser_parcels, get_glasser_CAatlas_mapping
glasser_CAatlas_mapping, CAatlas_names = get_glasser_CAatlas_mapping()

df_nets_counts = pd.DataFrame(columns=[int(net_i) for net_i in range(2, 10)], index=range(1, 67), dtype=float)
df_nets_counts.index.name = 'subject'
df_nets_counts = df_nets_counts.join(group_assignment).set_index(['group'], append=True)

cleaned = True #  with small pieces removed:
spec  = '_cleaned_fsaverage5' if cleaned else ''

for sub in subList:
    try:
        consensus_labels = np.load(op.join(net_folder,f'sub-{sub}_consensusMapping_confspec-36Pscrub3BPfilter{spec}.npy'))
        if cleaned == False: # only vertices within standard mask saved
            nets_fsav5 = np.full(mask.shape[0], np.nan, dtype=float)
            nets_fsav5[mask] = consensus_labels
            consensus_labels = nets_fsav5
    
        consensus_labels = consensus_labels[mask]
        net_i, counts = np.unique(consensus_labels, return_counts=True)

        for net, count in zip(net_i, counts):
            df_nets_counts.loc[int(sub), int(net)] = count

    except Exception as e:
        print(f'Error processing sub-{sub}: {e}')

df_nets = df_nets_counts.drop(-1, axis=1)
df_nets_renamed = df_nets.copy()
for net in df_nets.columns:
    df_nets_renamed.rename(columns={net: CAatlas_names.loc[net].iloc[0]}, inplace=True)

Error processing sub-02: [Errno 2] No such file or directory: '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk/derivatives/networks_infomap_full_01/sub-02_consensusMapping_confspec-36Pscrub3BPfilter_cleaned_fsaverage5.npy'


In [74]:
# convert to percentage of total vertices per subject
df_nets_pct = df_nets_renamed.div(df_nets_renamed.sum(axis=1), axis=0) * 100


In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu, normaltest
from statsmodels.stats.multitest import multipletests
import pandas as pd

alpha = 0.05
results = []

df = df_nets_pct
networks = df.columns[:8]

for net in networks:
    ctrl = df.xs(0, level='group')[net].dropna()
    dys  = df.xs(1, level='group')[net].dropna()

    ctrl_mean, ctrl_sd = ctrl.mean(), ctrl.std()
    dys_mean, dys_sd   = dys.mean(), dys.std()

    pooled = df[net].dropna()
    _, p_normal = normaltest(pooled)

    if p_normal > alpha:
        stat, pval = ttest_ind(ctrl, dys)
        test = "t"
        stat_label = f"t = {stat:.2f}"
    else:
        stat, pval = mannwhitneyu(ctrl, dys, alternative="two-sided")
        test = "MWU"
        stat_label = f"U = {stat:.0f}"

    results.append({
        "Network": net,
        "Control (% ± SD)": f"{ctrl_mean:.2f} ± {ctrl_sd:.2f}",
        "Dyscalculia (% ± SD)": f"{dys_mean:.2f} ± {dys_sd:.2f}",
        "Test": test,
        "Statistic": stat_label,
        "p_raw": pval
    })

df_results = pd.DataFrame(results)

# FDR correction
df_results["p_FDR"] = multipletests(df_results["p_raw"], method="fdr_bh")[1]

df_results["p (raw)"] = df_results["p_raw"].apply(format_p)
df_results["p (FDR)"] = df_results["p_FDR"].apply(format_p)

df_table = df_results[
    ["Network", "Control (% ± SD)", "Dyscalculia (% ± SD)",
     "Test", "Statistic", "p (raw)", "p (FDR)"]
]

df_table


,Network,Control (% ± SD),Dyscalculia (% ± SD),Test,Statistic,p (raw),p (FDR)
0,Visual2,14.31 ± 2.44,14.38 ± 2.44,MWU,U = 500,0.718,0.818
1,Somatomotor,18.05 ± 6.29,18.12 ± 7.81,MWU,U = 546,0.818,0.818
2,Cingulo-Opercular,14.64 ± 5.46,16.26 ± 5.92,t,t = -1.13,0.262,0.700
3,Dorsal-attention,17.57 ± 5.66,15.76 ± 6.03,t,t = 1.24,0.218,0.700
4,Language,2.04 ± 1.34,1.62 ± 1.03,t,t = 0.84,0.409,0.818
5,Frontoparietal,7.43 ± 4.48,8.68 ± 5.91,t,t = -0.64,0.528,0.818
6,Auditory,6.88 ± 3.37,6.61 ± 3.11,t,t = 0.32,0.747,0.818
7,Default,26.03 ± 3.97,23.79 ± 3.76,t,t = 2.34,0.022,0.179


In [77]:
df_table.to_csv(op.join(plots_folder, 'supTable_wholeBrain_PFMnetcount_groupComp.csv'), index=False)